# Search over token space of FlexTok
Here, I'll implement an automated search over the token space of FlexTok using LPIPS similarity as the objective.

In [1]:
# Switch path to root of project
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '5'
current_folder = globals()['_dh'][0]
os.chdir(os.path.dirname(os.path.abspath(current_folder)))

%load_ext autoreload
%autoreload 2

In [2]:
from PIL import Image
import matplotlib.pyplot as plt

import einops
import torch
import torchvision.transforms.functional as TF

from diffusers.models import AutoencoderKL

from flextok.flextok_wrapper import FlexTokFromHub, FlexTok
from flextok.utils.demo import imgs_from_urls, denormalize, batch_to_pil
from flextok.utils.misc import detect_bf16_support, get_bf16_context, get_generator

# The flag below controls whether to allow TF32 on matmul. This flag defaults to False in PyTorch 1.12 and later.
torch.backends.cuda.matmul.allow_tf32 = True
# The flag below controls whether to allow TF32 on cuDNN. This flag defaults to True.
torch.backends.cudnn.allow_tf32 = True

# Global no_grad
torch.set_grad_enabled(False)

# Automatically set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# Detect if bf16 is enabled or not
enable_bf16 = detect_bf16_support()
print('BF16 enabled:', enable_bf16)

Device: cuda
BF16 enabled: True


/home/iyu/miniconda3/envs/flextok/lib/python3.10/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


## 1 Loading example images

We load several demo images to showcase FlexTok and VAE reconstructions for. Feel free to use your own images!

When working with ImageNet-style images, we recommend loading the dedicated ImageNet FlexTok models, while for all other cases we recommend the DFN version.

In [3]:
from flextok.utils.dataloader import SyntheticFaceDataset, create_celeb_dataloader, CelebADataset, CelebAHQDataset
from flextok.utils.demo import denormalize, batch_to_pil

# load dataset
IMG_SIZE = 256
BATCH_SIZE = 4
DATASET_NAME = "celeba"  # "celeba" or "celebahq" or "synth_faces"
DATASET_PATH = f"./data/{DATASET_NAME}/"

# test dataset
val_dataset = CelebAHQDataset(
    root_dir=DATASET_PATH,
    img_size=IMG_SIZE,
    split="val",
)

print(f"val dataset: {len(val_dataset)} images")
# get images from val dataset
val_dataloader = create_celeb_dataloader(
    dataset_type=DATASET_NAME,
    root_dir=DATASET_PATH,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=1,
)
# get a batch of images
imgs = next(iter(val_dataloader)).to(device)

# imgs = imgs_from_urls([
#     "https://cdn.mos.cms.futurecdn.net/fAmJwNKXnajJb4aZYyvfsH.jpg",
#     "https://wolflawpllc.com/wp-content/uploads/2023/11/crowd-surge-at-a-concert.png",
#     "https://www.album-online.com/photos/prev/Njc2NjE1MA/album_alb5538449.jpg",
#     "https://imagez.tmz.com/image/6d/4by3/2022/04/18/6dad5b56b281408abd8b4178786ea708_md.png"
# ]).to(device)
# apply flips to test robustness
print('Loaded images:')
batch_to_pil(imgs)
secret_image = imgs[0:1]  # Select the first image as the "secret image" to be guessed

val dataset: 1500 images
Loaded images:


# 2. Load twenty questions API

In [4]:
from twenty_questions.twenty_questions import (
    load_flextok_model, 
    get_possible_combos, 
    sample_images_per_quantization,
    convert_images_to_pil,
    zhat_to_tokens,
    auto_twenty_q  # Import the new auto_twenty_q function
)
# load model
ckpt_path = "/home/iyu/ml-flextok/checkpoints/celeba_d18_arcface_fsq_8/20260108/checkpoint_best.pt"
new_levels = [8]  # set new levels for quantization
model = load_flextok_model(
    ckpt_path=ckpt_path,
    fsq_level=new_levels
).to(device)

Device: cuda
BF16 enabled: True
Overriding FSQ levels with: [8]
Adjusting encoder output dimension from 6 to 1
Adjusting decoder input dimension to 1


# 3. Load DreamSim

# 3a. Load VLM-based Similarity Scorer (Alternative to DreamSim)

You can use a Vision-Language Model (VLM) instead of DreamSim for more semantic understanding of facial identity and geometry.

Two options are available:
1. **VLMQASimilarityScorer**: Uses a VLM with detailed prompts to analyze facial identity and geometry (slower but more semantic)
2. **CLIPSimilarityScorer**: Uses CLIP embeddings for fast similarity comparison (faster, similar to DreamSim)

In [ ]:
from twenty_questions.vlm_scorer import VLMQASimilarityScorer, CLIPSimilarityScorer

# Option 1: VLM-based scorer with detailed facial analysis
# Supported models: 'llava-hf/llava-1.5-7b-hf', 'llava-hf/llava-1.5-13b-hf', 'Qwen/Qwen-VL-Chat'
vlm_scorer = VLMQASimilarityScorer(
    model_name='llava-hf/llava-1.5-7b-hf',
    device=device,
    load_in_4bit=True,  # Use 4-bit quantization to save memory
)

# Option 2: CLIP-based scorer (faster alternative)
# clip_scorer = CLIPSimilarityScorer(
#     model_name='ViT-B/16',  # or 'ViT-L/14' for better quality
#     device=device,
# )

# Test the VLM scorer with two images
print("Testing VLM scorer...")
test_img1 = convert_images_to_pil(secret_image)[0]
test_img2 = convert_images_to_pil(imgs[1:2])[0]

# This will return a distance score (lower = more similar)
test_score = vlm_scorer(test_img1, test_img2)
print(f"VLM similarity distance: {test_score:.4f}")
print("(Lower scores indicate higher similarity)")

In [13]:
from dreamsim import dreamsim
# load DreamSim model
dreamsim_model, dreamsim_preprocess = dreamsim(pretrained=True, device=device)

Using cached ./models


Using cache found in ./models/facebookresearch_dino_main


# 4. Loop for DreamSim-based 20Q

In [7]:
# Prepare tokens list for auto_twenty_q
# Get all possible quantization combinations
all_zhats = get_possible_combos(model).to(device)

# Convert zhats to tokens
tokens_list = zhat_to_tokens(model, all_zhats).unsqueeze(-1)
# DEBUG: change last one to [[1]] instead of [[2]]
tokens_list[-1] = torch.tensor([[new_levels[0] - 1]], device=tokens_list[-1].device)
tokens_list = list(tokens_list.split(1))
print(f"Tokens list (REPLACED LAST WITH [[{new_levels[0] - 1}]]):", [t.item() for t in tokens_list])
print("Total tokens to sample from:", len(tokens_list))

FSQ levels: tensor([8], device='cuda:0', dtype=torch.int32)
codebook size: 8
Total combinations (must equal codebook size): 8
tokens shape: torch.Size([8])
Tokens list (REPLACED LAST WITH [[7]]): [0, 1, 2, 3, 4, 5, 6, 7]
Total tokens to sample from: 8


# 5. Run eval model and plot choices

In [ ]:
# Run auto_twenty_q with GREEDY search (default)
chosen_history_greedy, rejected_history_greedy = auto_twenty_q(
    flextok_model=model,
    secret_image=convert_images_to_pil(secret_image)[0],
    tokens_list=tokens_list,
    num_samples_per_quantization=4,
    enable_bf16=enable_bf16,
    eval_model=dreamsim_model,
    num_questions=256,
    search_algorithm="greedy",  # either 'greedy' or 'beam'
    preprocess_fn=dreamsim_preprocess
)

Device: cuda
BF16 enabled: True
Generating 8 images in parallel (batch mode)...


/home/iyu/ml-flextok/flextok/model/utils/posembs.py:124: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
  posembs = posembs[slices]


Iteration 1: Top beam avg score 0.4722, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Iteration 2: Top beam avg score 0.4228, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Iteration 3: Top beam avg score 0.4030, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Iteration 4: Top beam avg score 0.3871, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Iteration 5: Top beam avg score 0.3700, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in paral

In [ ]:
import math

# Plot DreamSim score over time and show chosen images grid for Beam search
import matplotlib.pyplot as plt

# Extract tokens, scores, and images from chosen_history_greedy (which is actually beam search from cell above)
tokens = [int(item[0].item()) for item in chosen_history_greedy]
scores = [float(item[2]) for item in chosen_history_greedy]
images = [item[1] for item in chosen_history_greedy]  # PIL.Image.Image

# 1) Plot scores over iterations
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(scores) + 1), scores, marker='o', markersize=3, linewidth=1, color='blue')
min_idx = scores.index(min(scores))
plt.scatter(min_idx + 1, scores[min_idx], color='red', zorder=5)
plt.annotate(f"min: {scores[min_idx]:.3f}", (min_idx + 1, scores[min_idx]),
             xytext=(min_idx + 1, scores[min_idx] + 0.02),
             arrowprops=dict(arrowstyle="->", color="red"), fontsize=9)
plt.xlabel("Iteration")
plt.ylabel("DreamSim score (lower = more similar)")
plt.title("DreamSim score over iterations (Beam Search)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 2) Display chosen images in a grid labeled with their token and score
n = len(images)
ncols = 16
nrows = math.ceil(n / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2))
axes = axes.flatten()

for i in range(len(axes)):
    ax = axes[i]
    if i < n:
        ax.imshow(images[i])
        ax.set_title(f"iter:{i+1} tok:{tokens[i]}  {scores[i]:.3f}", fontsize=8)
        ax.axis("off")
    else:
        ax.axis("off")

plt.suptitle("Chosen images - Beam Search (labeled by token and DreamSim score)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# 6. Run with Beam Search

Now let's run the same task with beam search to compare the results. Beam search maintains multiple candidates at each step, which can lead to better global solutions.

In [ ]:
# Run auto_twenty_q with BEAM search
chosen_history_beam, rejected_history_beam = auto_twenty_q(
    flextok_model=model,
    secret_image=convert_images_to_pil(secret_image)[0],
    tokens_list=tokens_list,
    num_samples_per_quantization=4,
    enable_bf16=enable_bf16,
    eval_model=dreamsim_model,
    num_questions=256,
    search_algorithm="beam",  # Use beam search
    preprocess_fn=dreamsim_preprocess,
    beam_width=3  # Number of beams to maintain
)

Generating 8 images in parallel (batch mode)...
Iteration 1: Top beam avg score 0.4832, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Iteration 2: Top beam avg score 0.4227, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Iteration 3: Top beam avg score 0.3900, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Iteration 4: Top beam avg score 0.3757, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Generating 8 images in parallel (batch mode)...
Iteration 5: Top beam avg score 0.3681, maintaining 3 beams
Generating 8 images in parallel (batch mode)...
Generating 8 images in paral

In [ ]:
# Plot DreamSim score over time and show chosen images grid for BEAM search
import matplotlib.pyplot as plt

# Extract tokens, scores, and images from chosen_history_beam
tokens_beam = [int(item[0].item()) for item in chosen_history_beam]
scores_beam = [float(item[2]) for item in chosen_history_beam]
images_beam = [item[1] for item in chosen_history_beam]  # PIL.Image.Image

# 1) Plot scores over iterations
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(scores_beam) + 1), scores_beam, marker='o', markersize=3, linewidth=1, color='blue')
min_idx_beam = scores_beam.index(min(scores_beam))
plt.scatter(min_idx_beam + 1, scores_beam[min_idx_beam], color='red', zorder=5)
plt.annotate(f"min: {scores_beam[min_idx_beam]:.3f}", (min_idx_beam + 1, scores_beam[min_idx_beam]),
             xytext=(min_idx_beam + 1, scores_beam[min_idx_beam] + 0.02),
             arrowprops=dict(arrowstyle="->", color="red"), fontsize=9)
plt.xlabel("Iteration")
plt.ylabel("DreamSim score (lower = more similar)")
plt.title("DreamSim score over iterations (Beam Search)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 2) Display chosen images in a grid labeled with their token and score
n = len(images_beam)
ncols = 16
nrows = math.ceil(n / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2))
axes = axes.flatten()

for i in range(len(axes)):
    ax = axes[i]
    if i < n:
        ax.imshow(images_beam[i])
        ax.set_title(f"iter:{i+1} tok:{tokens_beam[i]}  {scores_beam[i]:.3f}", fontsize=8)
        ax.axis("off")
    else:
        ax.axis("off")

plt.suptitle("Chosen images - Beam Search (labeled by token and DreamSim score)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# 7. Compare Greedy vs Beam Search

In [ ]:
# Compare Greedy vs Beam Search results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot greedy scores
ax1.plot(range(1, len(scores) + 1), scores, marker='o', markersize=3, linewidth=1, color='green', label='Greedy')
ax1.set_xlabel("Iteration")
ax1.set_ylabel("DreamSim score (lower = more similar)")
ax1.set_title("Greedy Search")
ax1.grid(alpha=0.3)
ax1.legend()

# Plot beam scores
ax2.plot(range(1, len(scores_beam) + 1), scores_beam, marker='o', markersize=3, linewidth=1, color='blue', label='Beam')
ax2.set_xlabel("Iteration")
ax2.set_ylabel("DreamSim score (lower = more similar)")
ax2.set_title("Beam Search")
ax2.grid(alpha=0.3)
ax2.legend()

plt.suptitle("Comparison: Greedy vs Beam Search", fontsize=14)
plt.tight_layout()
plt.show()

# Print summary statistics
print("=" * 50)
print("SUMMARY STATISTICS")
print("=" * 50)
print(f"Greedy Search:")
print(f"  Final score: {scores[-1]:.4f}")
print(f"  Best score: {min(scores):.4f}")
print(f"  Average score: {sum(scores) / len(scores):.4f}")
print()
print(f"Beam Search (width={3}):")
print(f"  Final score: {scores_beam[-1]:.4f}")
print(f"  Best score: {min(scores_beam):.4f}")
print(f"  Average score: {sum(scores_beam) / len(scores_beam):.4f}")
print("=" * 50)

In [27]:
# Run automated 20Q with VLM scorer
# NOTE: This will be slower than DreamSim due to the VLM inference, 
# but should provide more semantic understanding of facial identity

chosen_history_vlm, rejected_history_vlm = auto_twenty_q(
    flextok_model=model,
    secret_image=convert_images_to_pil(secret_image)[0],
    num_samples_per_quantization=4,
    enable_bf16=enable_bf16,
    eval_model=vlm_scorer,  # Using VLM instead of DreamSim
    num_questions=256  # Run for all tokens
)

NameError: name 'vlm_scorer' is not defined